# <font color=firebrick>Econometrics — M2 MBFA</font>
## TD 5 / 6 — Panel Data on Real Stock Market Data

**Prerequisites:** Session 5 (CM) — panel data, fixed effects, the within transformation.

**Data source.** We use [`yfinance`](https://pypi.org/project/yfinance/), a free Python package that downloads real, **live** stock market data (prices, volumes) directly from Yahoo Finance - no account or API key needed. Because the data is live, the exact numbers you get will differ slightly from the ones printed in this notebook (and from one day to the next) - this is expected, and is itself a feature: it is real market data, not a given teaching dataset.

**Format:** Part 1 recaps classic finance measures on a single stock; Part 2 builds a real panel (several stocks × many days) and applies Session 5's fixed-effects toolkit; Part 3 repeats the analysis on a basket of tickers you choose.

**Today's objectives:**
- Know what each column of a Yahoo Finance download means, and compute the standard return/risk measures from it.
- Recognize that a basket of stocks observed daily *is* panel data, in exactly Session 5's sense.
- Apply pooled OLS, one-way, and two-way fixed effects to a real question: does trading volume predict next-day returns?

---


## Part 1 — Classic measures on a single stock

### 1.1 Getting the data, and understanding the columns

**Task 1.1.** Run the cell below. `Open`/`High`/`Low`/`Close` are self-explanatory (the price at market open, the day's high and low, and the closing price); `Volume` is the number of shares traded that day. We will mostly work with `Close`.


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.formula.api as smf

aapl = yf.download("AAPL", period="2y", progress=False)
aapl.columns = aapl.columns.get_level_values(0)  # drop the ticker level (single-ticker download)
aapl.head()


### 1.2 Returns: simple and log

**Task 1.2 - Part A (by hand).** A stock closes at $100$ on Monday, $104$ on Tuesday, and $98.80$ on Wednesday.
1. Compute the simple return for Tuesday, $r_t=(P_t-P_{t-1})/P_{t-1}$, and for Wednesday.
2. Compute the log return for Tuesday, $\log(P_t/P_{t-1})$, and for Wednesday.
3. Compute the two-day **cumulative** simple return (Monday to Wednesday) two ways: (i) directly from $P_{\text{Mon}}$ and $P_{\text{Wed}}$, (ii) by compounding the two daily simple returns, $(1+r_{\text{Tue}})(1+r_{\text{Wed}})-1$. Do they match? Would summing the two daily *log* returns instead also give the right two-day log return?

<details>
<summary>Show answer</summary>

Tuesday: simple $r=(104-100)/100=0.04$ ($4\%$); log $=\log(104/100)\approx0.0392$.

Wednesday: simple $r=(98.8-104)/104=-0.05$ ($-5\%$); log $=\log(98.8/104)\approx-0.0513$.

Cumulative simple return: directly, $(98.8-100)/100=-0.012$ ($-1.2\%$); by compounding, $(1.04)(0.95)-1=-0.012$ - they match (simple returns compound multiplicatively, never additively). Log returns, by contrast, **add**: $0.0392+(-0.0513)=-0.0121\approx\log(98.8/100)$ - this additivity is precisely why log returns are often preferred for multi-period analysis.

</details>

**Task 1.2 - Part B (computational).** Complete the code below to compute both return series for `aapl`, and check your formula against pandas' own `pct_change()`.


In [ ]:
# TODO: compute the simple return and the log return of the Close price
aapl['ret'] = ...
aapl['logret'] = ...

assert np.allclose(aapl['ret'], aapl['Close'].pct_change(), equal_nan=True), "Your simple return doesn't match pandas' pct_change() - check your formula."
print("Self-check passed!")

aapl = aapl.dropna(subset=['ret', 'logret'])
aapl[['Close', 'ret', 'logret']].head()


### 1.3 Volatility

**Task 1.3.** The standard measure of risk is the standard deviation of returns. Daily volatility is rarely reported on its own - it is conventionally **annualized** by multiplying by $\sqrt{252}$ (the typical number of trading days in a year), since variance scales with time under the usual independence assumptions. Complete the code below.


In [ ]:
daily_vol = aapl['ret'].std()

# TODO: annualize the daily volatility
annual_vol = ...

print(f"Daily volatility:    {daily_vol:.4f}")
print(f"Annualized volatility: {annual_vol:.2%}")
assert np.isclose(annual_vol, daily_vol * np.sqrt(252)), "Check the annualization formula (daily_vol * sqrt(252))."
print("Self-check passed!")


### 1.4 Beta: the market-sensitivity regression

**Task 1.4.** The CAPM beta of a stock is *literally* the OLS slope coefficient (Session 1-2) from regressing the stock's return on the market's return, $r_{i,t}=\alpha+\beta\,r_{m,t}+\varepsilon_t$. We use the S&P 500 (`^GSPC`) as the market proxy. Interpret $\hat\beta$: is this stock more or less volatile than the market as a whole?


In [ ]:
mkt = yf.download("^GSPC", period="2y", progress=False)
mkt.columns = mkt.columns.get_level_values(0)
mkt_ret = mkt['Close'].pct_change().dropna()

beta_data = pd.DataFrame({'stock': aapl['ret'], 'market': mkt_ret}).dropna()
beta_model = smf.ols('stock ~ market', data=beta_data).fit()
print(beta_model.summary())


### 1.5 Sharpe ratio

**Task 1.5.** The Sharpe ratio measures risk-adjusted return: $\dfrac{\text{mean excess return}}{\text{volatility of return}}$, usually annualized. For simplicity, take the risk-free rate as $0$ here (a reasonable approximation for a short comparison). Complete the code below.


In [ ]:
mean_daily_ret = aapl['ret'].mean()

# TODO: compute the annualized Sharpe ratio: (mean_daily_ret * 252) / (daily_vol * sqrt(252))
sharpe = ...
print(f"Annualized Sharpe ratio: {sharpe:.2f}")


## Part 2 — A real panel: does trading volume predict returns?

We now download a basket of stocks, reshape the result into panel form (ticker $\times$ date, exactly Session 5's structure), and ask: does a stock's trading volume on one day predict its return on the *next* day? A popular trading intuition says unusually high volume signals an incoming move - let's test it properly.

### 2.1 Building the panel

**Task 2.1.** Run the cell below. `melt()` reshapes a wide table (one column per ticker) into a long/panel table (one row per ticker-date pair) - the standard shape for panel regressions.


In [ ]:
tickers = ["AAPL", "MSFT", "JPM", "XOM", "KO", "PG", "DIS", "NKE"]
data = yf.download(tickers, period="2y", progress=False)

long_close = data['Close'].reset_index().melt(id_vars='Date', var_name='ticker', value_name='close')
long_volume = data['Volume'].reset_index().melt(id_vars='Date', var_name='ticker', value_name='volume')
panel = long_close.merge(long_volume, on=['Date', 'ticker']).sort_values(['ticker', 'Date']).reset_index(drop=True)

print(f"{panel['ticker'].nunique()} tickers x {panel['Date'].nunique()} trading days = {len(panel)} rows")
panel.head()


### 2.2 Building the variables

**Task 2.2.** Complete the code below:
1. `ret`: the daily return, computed **within each ticker** (use `.groupby('ticker')['close'].pct_change()` - exactly as you would demean *within* each individual in Session 5, we must compute returns *within* each ticker, never across two different stocks).
2. `lvolume`: the log of volume (trading volume is heavily right-skewed - recall Session 1's discussion of when to use logs).
3. `lvolume_lag`: **yesterday's** log-volume for that same ticker (`.groupby('ticker')['lvolume'].shift(1)`) - we want volume to *predict* tomorrow's return, not be measured on the same day.


In [ ]:
# TODO: compute the three variables described above
panel['ret'] = ...
panel['lvolume'] = ...
panel['lvolume_lag'] = ...

panel_reg = panel.dropna(subset=['ret', 'lvolume_lag']).copy()
assert panel_reg.groupby('ticker')['ret'].apply(lambda s: s.notna().all()).all(), "ret should be computed within each ticker - check your groupby."
print("Self-check passed! Regression sample:", len(panel_reg), "observations.")


### 2.3 Pooled OLS, then fixed effects

**Task 2.3.** Estimate three specifications, exactly as in Session 5's `wagepan` example: (1) pooled OLS, (2) ticker fixed effects only, (3) two-way (ticker **and** date) fixed effects. `C(Date)` creates one dummy per trading day - a lot of columns, but Session 5 already showed this runs in well under a second.


In [ ]:
pooled = smf.ols('ret ~ lvolume_lag', data=panel_reg).fit()
fe_ticker = smf.ols('ret ~ lvolume_lag + C(ticker)', data=panel_reg).fit()
fe_two_way = smf.ols('ret ~ lvolume_lag + C(ticker) + C(Date)', data=panel_reg).fit()

print("Pooled:          coef =", pooled.params['lvolume_lag'], " p =", pooled.pvalues['lvolume_lag'])
print("Ticker FE:       coef =", fe_ticker.params['lvolume_lag'], " p =", fe_ticker.pvalues['lvolume_lag'])
print("Two-way FE:      coef =", fe_two_way.params['lvolume_lag'], " p =", fe_two_way.pvalues['lvolume_lag'])


**Task 2.4 (interpretation - the main point of this TD).** Across all three specifications, is `lvolume_lag` statistically significant at the 5% level? If your result matches the usual finding here (it should, but markets move - check your own p-values), **this is not a failure of the method**: a *non-result* is itself informative. If yesterday's volume reliably predicted tomorrow's return, that pattern would be exploitable, and traders exploiting it would push prices to erase it - this is the logic of **market efficiency**. Contrast this with Session 5's `wagepan` example, where adding fixed effects **changed** the conclusion (the union premium shrank by half): here, the conclusion is expected to be stable across specifications - there was no confounding to remove, because there was no effect to begin with. Both outcomes - "fixed effects changed everything" and "fixed effects changed nothing" - are informative, for different reasons.

### 2.4 Checking the two routes agree

**Task 2.5.** As in Session 5, confirm that the ticker fixed-effects coefficient above matches what you would get from the explicit within (demeaning) transformation - the two routes are still equivalent here, on real data.


In [ ]:
demean_cols = ['ret', 'lvolume_lag']
means = panel_reg.groupby('ticker')[demean_cols].transform('mean')
panel_dm = panel_reg.copy()
for c in demean_cols:
    panel_dm[c + '_dm'] = panel_dm[c] - means[c]

fe_manual = smf.ols('ret_dm ~ lvolume_lag_dm - 1', data=panel_dm).fit()
print("Within transform coef:", fe_manual.params['lvolume_lag_dm'])
print("C(ticker) FE coef:    ", fe_ticker.params['lvolume_lag'])
assert np.isclose(fe_manual.params['lvolume_lag_dm'], fe_ticker.params['lvolume_lag'], atol=1e-6), "The two routes should give the identical coefficient."
print("Self-check passed!")


## Part 3 — Your own basket

Pick $5$ to $10$ tickers yourself - a sector you're interested in, a mix of large and small caps, European tickers (Yahoo Finance suffixes, e.g. `MC.PA` for LVMH on Euronext Paris, `^FCHI` for the CAC 40 as a market proxy), whatever you like. Repeat the Part 2 analysis, step by step.

**Step 1 - Pick your tickers and download the data.**


**Step 2 - Build the panel** (melt to long format, as in 2.1).

**Step 3 - Build `ret`, `lvolume`, and `lvolume_lag`** (as in 2.2) - or substitute a question of your own (e.g., does a stock's volatility predict next-day volume?).

**Step 4 - Estimate pooled OLS, ticker FE, and two-way FE.**

**Step 5 - Interpret.** Does your result look like Section 2.3/2.4 (no robust effect, consistent with market efficiency) or does it change meaningfully once fixed effects are added (like `wagepan` in Session 5)? Either way, explain *why* you think that is, given the tickers you chose.

---

## A note on dependencies

This notebook uses `yfinance`, which is not used anywhere else in the course. If you are running this on Binder and it fails to import, make sure `yfinance` has been added to the course's `requirements.txt` on GitHub.
